In [1]:
import os
import re
import heapq
from collections import defaultdict, Counter
from typing import List, Dict, Tuple
import nltk
from nltk.corpus import stopwords
from navec import Navec
from razdel import tokenize
from natasha import Segmenter, NewsEmbedding, NewsMorphTagger, MorphVocab, Doc

In [2]:
class TextPreprocessor:
    """Предобработка текста"""
    def __init__(self):
        self.stop_words_ru = set(stopwords.words('russian'))
        self.segmenter = Segmenter()
        self.emb = NewsEmbedding()
        self.morph_tagger = NewsMorphTagger(self.emb)
        self.morph_vocab = MorphVocab()
        self.url_pattern = re.compile(r'http\S+|www\S+|https\S+')
        self.non_alpha_pattern = re.compile(r'[^а-яА-ЯёЁ\s]')
        self.whitespace_pattern = re.compile(r'\s+')

    def clean_text(self, text):
        """Очистка текста"""
        if not isinstance(text, str) or text.strip() == "":
            return ""
        text = self.url_pattern.sub('', text)
        text = self.non_alpha_pattern.sub(' ', text)
        text = text.lower().strip()
        text = self.whitespace_pattern.sub(' ', text)
        return text

    def remove_stopwords(self, tokens):
        """Удаление стоп-слов"""
        return [word for word in tokens if word not in self.stop_words_ru]

    def lemmatize_text(self, tokens):
        """Лемматизация токенов"""
        doc = Doc(" ".join(tokens))
        doc.segment(self.segmenter)
        doc.tag_morph(self.morph_tagger)
        for token in doc.tokens:
            token.lemmatize(self.morph_vocab)
        return [token.lemma for token in doc.tokens]

    def preprocess(self, text):
        """Полный пайплайн предобработки"""
        cleaned_text = self.clean_text(text)
        if not cleaned_text:
            return ""
        tokenized_text = [t.text for t in tokenize(cleaned_text)]
        filtered_text = self.remove_stopwords(tokenized_text)
        lemmatized_text = self.lemmatize_text(filtered_text)
        return " ".join(lemmatized_text)

In [3]:
files_dir = "/content/"

In [4]:
txt_files = [i for i in os.listdir(files_dir) if i.endswith(".txt")]

In [5]:
txt_files

['Токенизация_2.txt',
 'Токенизация_3.txt',
 'Токенизация_4.txt',
 'Токенизация_1.txt']

In [6]:
texts = []
for file in sorted(txt_files):
    with open(os.path.join(files_dir, file), "r", encoding="utf-8") as f:
        texts.append(f.read())

In [7]:
texts

['Серотонин прежде всего известен как нейромедиатор. Однако эта молекула может вступать в реакции — например, присоединяться к клеточным и внеклеточным белкам, то есть серотонилировать их. Ранее было показано, что серотонилированию подвержены в том числе гистоны — основные белки хроматина, которые не только обеспечивают упаковку ДНК, но и регулируют активность генов. Как серотонилирование связано с экспрессией генов, изучено достаточно плохо, и еще хуже изучены соответствующие молекулярно-физиологические процессы. Исследователи из США показали, что серотонин регулирует развитие опухолей головного мозга, ограничивая экспрессию ключевых транскрипционных факторов и влияя на активность нейронов, которые окружают эти опухоли. Это первая работа, описывающая подобный механизм поддержания злокачественных новообразований.\n\nВ поиске способов лечения злокачественных опухолей исследователи изучают не только ее саму, но и окружающие ее здоровые ткани. Буквально в прошлом году журнал Nature опубли

In [8]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
preprocessor = TextPreprocessor()

In [10]:
processed_texts = [preprocessor.preprocess(text) for text in texts]

In [11]:
processed_texts

['серотонин прежде известный нейромедиатор однако этот молекула вступать реакция например присоединяться клеточный внеклеточный белок серотонилировать ранее показать серотонилирование подверженный число гистон основной белок хроматин который обеспечивать упаковка днк регулировать активность ген серотонилирование связать экспрессия ген изучить достаточно плохо плохой изучить соответствующий молекулярный физиологический процесс исследователь сша показать серотонин регулировать развитие опухоль головной мозг ограничивать экспрессия ключевой транскрипционный фактор влиять активность нейрон который окружать опухоль это первый работа описывать подобный механизм поддержание злокачественный новообразование поиск способ лечение злокачественный опухоль исследователь изучать сам окружать здоровый ткань буквально прошлый год журнал опубликовать исследование который показать глиобластома изменять нейронный путь пациент приводить когнитивный проблема снижение выживаемость нейронный активность также 

In [25]:
class BPETokenizer:
    def __init__(self, vocab_size: int, min_frequency: int = 2):
        self.vocab_size = vocab_size
        self.min_frequency = min_frequency
        self.merges = {}
        self.vocab = []
        self.word_cache = {}  # Кэш для уже токенизированных слов

    def get_stats(self, words: dict) -> dict:
        # Подсчет частот пар символов/токенов
        pairs = defaultdict(int)
        for word, freq in words.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i + 1])] += freq
        return pairs

    def merge_vocab(self, pair: tuple, words: dict) -> dict:
        # Слияние пары в словаре
        new_words = {}
        bigram = ' '.join(pair)
        replacement = ''.join(pair)

        for word, freq in words.items():
            parts = word.split()
            i = 0
            new_parts = []

            while i < len(parts):
                if i < len(parts) - 1 and parts[i] == pair[0] and parts[i + 1] == pair[1]:
                    new_parts.append(replacement)
                    i += 2
                else:
                    new_parts.append(parts[i])
                    i += 1

            new_word = ' '.join(new_parts)
            new_words[new_word] = freq

        return new_words

    def train(self, word_freqs: dict) -> None:
        # Преобразуем слова в списки отдельных символов
        words = {' '.join(word): freq for word, freq in word_freqs.items()}

        # Инициализируем словарь начальными символами
        unique_chars = set()
        for word in words:
            for char in word.split():
                unique_chars.add(char)
        self.vocab = list(unique_chars)

        # Выполняем слияния до достижения нужного размера словаря
        merge_id = 0
        while len(self.vocab) < self.vocab_size:
            pairs = self.get_stats(words)
            if not pairs:
                break

            # Находим самую частую пару
            best_pair = max(pairs.items(), key=lambda item: item[1])
            if best_pair[1] < self.min_frequency:
                break

            # Сохраняем слияние и добавляем в словарь
            self.merges[best_pair[0]] = merge_id
            new_token = ''.join(best_pair[0])
            self.vocab.append(new_token)

            # Применяем слияние ко всем словам
            words = self.merge_vocab(best_pair[0], words)
            merge_id += 1

            # print(f"Слияние {merge_id}: {best_pair[0]} -> {new_token} (частота: {best_pair[1]})")

        print(f"Обучение завершено. Размер словаря: {len(self.vocab)}")

    def tokenize(self, word: str) -> list:
        # Проверяем кэш
        if word in self.word_cache:
            return self.word_cache[word]

        # Разбиваем слово на отдельные символы
        tokens = list(word)
        word_str = ' '.join(tokens)

        # Применяем слияния в порядке их добавления
        for pair, merge_id in sorted(self.merges.items(), key=lambda x: x[1]):
            bigram = ' '.join(pair)
            replacement = ''.join(pair)

            parts = word_str.split()
            i = 0
            new_parts = []

            while i < len(parts):
                if i < len(parts) - 1 and parts[i] == pair[0] and parts[i + 1] == pair[1]:
                    new_parts.append(replacement)
                    i += 2
                else:
                    new_parts.append(parts[i])
                    i += 1

            word_str = ' '.join(new_parts)

        # Сохраняем в кэш и возвращаем результат
        result = word_str.split()
        self.word_cache[word] = result
        return result

In [26]:
word_freqs = defaultdict(int)
for text in processed_texts:
    for word in text.split():
        word_freqs[word] += 1

In [27]:
bpe_tokenizer = BPETokenizer(vocab_size=5000, min_frequency=2)
bpe_tokenizer.train(word_freqs)

Обучение завершено. Размер словаря: 1393


In [33]:
tokenized_texts = []
for i, text in enumerate(processed_texts):
    tokenized_words = []
    for word in text.split():
        tokens = bpe_tokenizer.tokenize(word)
        tokenized_words.extend(tokens)
    tokenized_texts.append(tokenized_words)

    # Выводим первые несколько токенов каждого текста
    print(f"\nТекст #{i+1} (первые 30 токенов):")
    print(tokenized_words[:30])


Текст #1 (первые 30 токенов):
['серотонин', 'пре', 'ж', 'де', 'известный', 'нейромедиатор', 'однако', 'этот', 'молекула', 'вступать', 'ре', 'ак', 'ция', 'например', 'присоединяться', 'клеточный', 'внеклеточный', 'белок', 'серотони', 'лировать', 'ранее', 'показать', 'серотонилирование', 'под', 'вер', 'же', 'нный', 'число', 'гистон', 'осно']

Текст #2 (первые 30 токенов):
['хоанофлагеллят', 'воротничковый', 'жгутиконосец', 'это', 'своеобразный', 'группа', 'проти', 'сто', 'вый', 'который', 'родственный', 'животное', 'гриб', 'весь', 'ви', 'димо', 'сть', 'давно', 'именно', 'колониальный', 'воротничковый', 'жгутиконосец', 'дать', 'начало', 'первый', 'этот', 'многоклеточный', 'однако', 'именно', 'оставаться']

Текст #3 (первые 30 токенов):
['органический', 'вещество', 'содержаться', 'метеорит', 'являться', 'источник', 'информация', 'возможный', 'путь', 'за', 'рождение', 'жизнь', 'земля', 'химический', 'процесс', 'происходить', 'ранний', 'ста', 'д', 'ия', 'формирование', 'солнечный', 'система